# Unsupervised Learning
Clustering, dimensionality reduction and anomaly detection using scikit-learn.

## 1. Imports and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)
X = StandardScaler().fit_transform(X)

def plot_clusters(X, labels, title):
    plt.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=20)
    plt.title(title)
    plt.show()

## 2. K-Means

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)

plot_clusters(X, labels, "K-Means")
print("Silhouette:", silhouette_score(X, labels))
print("ARI:", adjusted_rand_score(y_true, labels))

### Choosing K: Elbow and Silhouette

In [ ]:
ks = range(2, 10)
inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ks, inertia, "o-"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(ks, sil, "o-"); ax[1].set_title("Silhouette score")
plt.show()

## 3. Hierarchical Clustering

In [ ]:
Z = linkage(X, method="ward")
plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.title("Dendrogram")
plt.show()

agg = AgglomerativeClustering(n_clusters=4, linkage="ward")
labels = agg.fit_predict(X)
plot_clusters(X, labels, "Agglomerative Clustering")
print("Silhouette:", silhouette_score(X, labels))

## 4. DBSCAN
Finds clusters of any shape and marks noise as `-1`.

In [ ]:
X_moons, _ = make_moons(n_samples=400, noise=0.07, random_state=42)

db = DBSCAN(eps=0.2, min_samples=5)
labels = db.fit_predict(X_moons)

plot_clusters(X_moons, labels, "DBSCAN on moons data")
print("Clusters:", len(set(labels)) - (1 if -1 in labels else 0))
print("Noise points:", (labels == -1).sum())

### K-Means vs DBSCAN on non-spherical data

In [ ]:
km_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, s=20); ax[0].set_title("K-Means")
ax[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels, s=20); ax[1].set_title("DBSCAN")
plt.show()

## 5. Gaussian Mixture Model
Soft clustering: each point gets a probability for every cluster.

In [ ]:
gmm = GaussianMixture(n_components=4, random_state=42)
labels = gmm.fit_predict(X)
probs = gmm.predict_proba(X)

plot_clusters(X, labels, "Gaussian Mixture")
print("BIC:", gmm.bic(X))
print("Probabilities of first point:", probs[0].round(3))

## 6. Mean Shift
Number of clusters is found automatically.

In [ ]:
ms = MeanShift(bandwidth=0.8)
labels = ms.fit_predict(X)

plot_clusters(X, labels, "Mean Shift")
print("Clusters found:", len(np.unique(labels)))

## 7. PCA (Dimensionality Reduction)

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

pca = PCA()
pca.fit(X_iris)
print("Explained variance:", pca.explained_variance_ratio_.round(3))

plt.plot(np.cumsum(pca.explained_variance_ratio_), "o-")
plt.xlabel("Components"); plt.ylabel("Cumulative variance")
plt.title("PCA explained variance")
plt.show()

X_pca = PCA(n_components=2).fit_transform(X_iris)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=iris.target, cmap="viridis")
plt.title("Iris in 2D (PCA)")
plt.show()

## 8. t-SNE (Visualization)

In [ ]:
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_iris)

plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=iris.target, cmap="viridis")
plt.title("Iris in 2D (t-SNE)")
plt.show()

## 9. Clustering on Real Data (Iris)

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = km.fit_predict(X_iris)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="viridis")
plt.title("K-Means clusters shown on PCA axes")
plt.show()

print("ARI vs true species:", adjusted_rand_score(iris.target, labels))
print(pd.crosstab(iris.target, labels, rownames=["Species"], colnames=["Cluster"]))

## 10. Anomaly Detection: Isolation Forest

In [ ]:
rng = np.random.RandomState(42)
X_normal = rng.normal(0, 1, (300, 2))
X_outliers = rng.uniform(-6, 6, (15, 2))
X_all = np.vstack([X_normal, X_outliers])

iso = IsolationForest(contamination=0.05, random_state=42)
pred = iso.fit_predict(X_all)

plt.scatter(X_all[:, 0], X_all[:, 1], c=pred, cmap="coolwarm", s=20)
plt.title("Isolation Forest (red = normal, blue = anomaly)")
plt.show()
print("Anomalies detected:", (pred == -1).sum())

## 11. Compare Clustering Algorithms

In [ ]:
models = {
    "KMeans": KMeans(n_clusters=4, n_init=10, random_state=42),
    "Agglomerative": AgglomerativeClustering(n_clusters=4),
    "GMM": GaussianMixture(n_components=4, random_state=42),
    "MeanShift": MeanShift(bandwidth=0.8),
}

rows = []
for name, model in models.items():
    labels = model.fit_predict(X)
    rows.append({
        "Model": name,
        "Clusters": len(np.unique(labels)),
        "Silhouette": silhouette_score(X, labels),
        "ARI": adjusted_rand_score(y_true, labels),
    })

pd.DataFrame(rows).round(3)